In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
import os

def fast_unzip(zip_path, extract_to):
    print(f'Đang giải nén {zip_path}...')
    if not os.path.exists(extract_to):
        os.makedirs(extract_to)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f'Hoàn thành giải nén vào {extract_to}')

# Đường dẫn tệp
train_zip = "/content/drive/My Drive/AIO_Homework/dence representation/data/data_train.zip"
test_zip = "/content/drive/My Drive/AIO_Homework/dence representation/data/data_test.zip"
dataset_path = "/content/dataset/"

# Chạy giải nén
if os.path.exists(train_zip):
    fast_unzip(train_zip, dataset_path)
else:
    print('Không tìm thấy tệp zip train. Kiểm tra lại đường dẫn!')

if os.path.exists(test_zip):
    fast_unzip(test_zip, dataset_path)
else:
    print('Không tìm thấy tệp zip test.')

Đang giải nén /content/drive/My Drive/AIO_Homework/dence representation/data/data_train.zip...
Hoàn thành giải nén vào /content/dataset/
Đang giải nén /content/drive/My Drive/AIO_Homework/dence representation/data/data_test.zip...
Hoàn thành giải nén vào /content/dataset/


In [4]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from tqdm import tqdm

In [5]:
# Tải file .bin pretrained tiếng Việt
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.bin.gz
!gunzip cc.vi.300.bin.gz

print("Đã tải xong FastText pretrained!")

--2026-04-21 02:41:44--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.33, 13.249.182.81, 13.249.182.62, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4504817955 (4.2G) [application/octet-stream]
Saving to: ‘cc.vi.300.bin.gz’

cc.vi.300.bin.gz    100%[===================>]   4.20G  57.0MB/s    in 78s     

2026-04-21 02:43:03 (55.0 MB/s) - ‘cc.vi.300.bin.gz’ saved [4504817955/4504817955]

Đã tải xong FastText pretrained!


In [7]:

import fasttext
import fasttext.util
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [8]:
ft = fasttext.load_model('cc.vi.300.bin')
print(f"Embedding dim: {ft.get_dimension()}")  # 300

Embedding dim: 300


In [9]:
def load_corpus_with_labels(directory):
    texts, labels = [], []
    for label_idx, label in enumerate(['neg', 'pos']):  # 0=neg, 1=pos
        subdir = os.path.join(directory, label)
        if not os.path.exists(subdir):
            continue
        for filename in os.listdir(subdir):
            filepath = os.path.join(subdir, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                texts.append(f.read())
                labels.append(label_idx)
    return texts, labels

train_texts, train_labels = load_corpus_with_labels("/content/dataset/data_train/train")
val_texts,   val_labels   = load_corpus_with_labels("/content/dataset/data_train/test")
test_texts,  test_labels  = load_corpus_with_labels("/content/dataset/data_test/test")

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

Train: 30000 | Val: 10000 | Test: 10000


In [16]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=300, hidden_dim=256, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
def make_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)

In [18]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mlp_model = MLPClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=1e-3)

EPOCHS  = 20
PATIENCE = 5
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    # Train
    mlp_model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for X_batch, y_batch in pbar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out  = mlp_model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (out.argmax(1) == y_batch).sum().item()
        train_total   += y_batch.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc  = train_correct / train_total

    # Validation
    mlp_model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out  = mlp_model(X_batch)
            loss = criterion(out, y_batch)
            val_loss    += loss.item()
            val_correct += (out.argmax(1) == y_batch).sum().item()
            val_total   += y_batch.size(0)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc  = val_correct / val_total

    pbar.set_postfix_str(
        f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}"
    )
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(mlp_model.state_dict(), "best_mlp.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered!")
            break

Epoch 1/20: 100%|██████████| 469/469 [00:01<00:00, 234.93it/s]


Train Loss: 0.4725 | Train Acc: 0.7791 | Val Loss: 0.4128 | Val Acc: 0.8170


Epoch 2/20: 100%|██████████| 469/469 [00:01<00:00, 433.36it/s]


Train Loss: 0.4028 | Train Acc: 0.8248 | Val Loss: 0.3959 | Val Acc: 0.8271


Epoch 3/20: 100%|██████████| 469/469 [00:01<00:00, 414.07it/s]


...

Train Loss: 0.3385 | Train Acc: 0.8560 | Val Loss: 0.3603 | Val Acc: 0.8458


Epoch 18/20: 100%|██████████| 469/469 [00:01<00:00, 333.18it/s]


Train Loss: 0.3320 | Train Acc: 0.8577 | Val Loss: 0.3590 | Val Acc: 0.8499


Epoch 19/20: 100%|██████████| 469/469 [00:01<00:00, 279.15it/s]


Train Loss: 0.3305 | Train Acc: 0.8596 | Val Loss: 0.3637 | Val Acc: 0.8468
Early stopping triggered!


In [19]:
mlp_model.load_state_dict(torch.load("best_mlp.pt"))
mlp_model.eval()

test_loss, test_correct, test_total = 0, 0, 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        out  = mlp_model(X_batch)
        loss = criterion(out, y_batch)
        test_loss    += loss.item()
        test_correct += (out.argmax(1) == y_batch).sum().item()
        test_total   += y_batch.size(0)

print(f"Test Loss: {test_loss/len(test_loader):.4f} | Test Acc: {test_correct/test_total:.4f}")

Test Loss: 0.3529 | Test Acc: 0.8555
